# 🤖 VGA PoC - Baseline Exploration on Kaggle: LIBERO & SmolVLA-450M

**Objective**:
1. Ingest the **LIBERO-Spatial** robotics manipulation dataset using Hugging Face's `lerobot` library.
2. Filter for our **3 target benchmark tasks**.
3. Inspect visual observations (RGB camera), robot proprioception, and 7D actions ($[\Delta x, \Delta y, \Delta z, r_x, r_y, r_z, \text{gripper}]$).
4. Load and inspect the existing baseline technology: **`lerobot/smolvla_base` (450M)**.
5. Compute the empirical normalization vectors ($\mu, \sigma, \sigma_{pos}^2, \sigma_{rot}^2$) and export `action_stats.json` for training our VGA model.

### Step 1: Install LeRobot and Dependencies

In [ ]:
# 1. Install av (PyAV video decoder), datasets, and num2words FIRST
!pip install --quiet av datasets num2words

# 2. Install LeRobot directly from Hugging Face source
!pip install --quiet git+https://github.com/huggingface/lerobot.git

# 3. Supporting ML and visualization packages
!pip install --quiet zarr einops scipy torchvision peft accelerate matplotlib

### Step 2: Verify GPU Accelerator

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU. In Kaggle, enable GPU under 'Settings' -> 'Accelerator' (choose T4 x 2 or P100).")

### Step 3: Fetch the LIBERO-Spatial Dataset via LeRobot

In [ ]:
# 1. Upstream patch: fix av.option.Option type annotation bug in modern LeRobot pyav_utils.py
import av
import types
if not hasattr(av, "option"):
    av.option = types.ModuleType("av.option")
    av.option.Option = object

# 2. Robust import supporting both new and legacy LeRobot package layouts
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
except ModuleNotFoundError as e:
    if "lerobot.datasets" in str(e):
        from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
    else:
        raise

# 3. Load LIBERO-Spatial dataset
repo_id = "lerobot/libero_spatial_image"
print(f"Fetching dataset '{repo_id}' from Hugging Face Hub...")

try:
    dataset = LeRobotDataset(repo_id)
    print(f"Successfully loaded {repo_id}!")
    print(f"- Number of episodes: {dataset.num_episodes}")
    print(f"- Total frame count: {dataset.num_frames}")
    print(f"- FPS: {dataset.fps}")
    print(f"- Available features: {list(dataset.features.keys())}")
except Exception as e:
    print(f"Notice: Failed with {repo_id}: {e}")
    print("Trying fallback repository 'lerobot/libero_spatial'...")
    repo_id = "lerobot/libero_spatial"
    dataset = LeRobotDataset(repo_id)
    print(f"Successfully loaded {repo_id}!")
    print(f"- Number of episodes: {dataset.num_episodes}")
    print(f"- Total frame count: {dataset.num_frames}")
    print(f"- FPS: {dataset.fps}")
    print(f"- Available features: {list(dataset.features.keys())}")

### Step 4: Filter the 3 Benchmark Tasks for our PoC

In [ ]:
TARGET_TASKS = [
    "pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate",
    "pick_up_the_alphabet_soup_and_place_it_in_the_basket",
    "push_the_plate_to_the_front_of_the_stove"
]

print("PoC Target Benchmark Tasks:")
for i, task in enumerate(TARGET_TASKS, 1):
    print(f"{i}. {task}")

# Inspect tasks available in the dataset metadata
if hasattr(dataset, 'meta') and hasattr(dataset.meta, 'tasks'):
    available_tasks = dataset.meta.tasks
    print(f"\nTotal unique tasks in dataset: {len(available_tasks)}")
    print("\nAll 10 Tasks in the Dataset:")
    for idx, t in enumerate(available_tasks):
        print(f"  [{idx}] {t}")
    
    print("\nMatching against PoC targets:")
    for t in TARGET_TASKS:
        t_clean = t.replace("_", " ")
        match = [task for task in available_tasks if t_clean in str(task) or t in str(task)]
        print(f"  Target '{t[:35]}...' matched: {match}")

### Step 5: Visualize a Step Sample (Camera Views & Robot Action)

In [ ]:
import matplotlib.pyplot as plt

# Sample the very first frame of the dataset
sample = dataset[0]

print("Sample keys:", sample.keys())
action = sample["action"]
print(f"Action vector shape: {action.shape}")
print(f"Action values [dx, dy, dz, rx, ry, rz, gripper]:\n{action.tolist()}")

# Identify and display image observations
image_keys = [k for k in sample.keys() if "image" in k]
print(f"Image observation streams: {image_keys}")

fig, axes = plt.subplots(1, len(image_keys), figsize=(5 * len(image_keys), 5))
if len(image_keys) == 1:
    axes = [axes]

for ax, img_k in zip(axes, image_keys):
    img_tensor = sample[img_k]
    # Convert from [C, H, W] to [H, W, C]
    if img_tensor.shape[0] in [1, 3]:
        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    else:
        img_np = img_tensor.cpu().numpy()
    ax.imshow(img_np)
    ax.set_title(img_k)
    ax.axis("off")

plt.tight_layout()
plt.show()

### Step 6: Compute Empirical Normalization Vectors
We calculate the mean $\boldsymbol{\mu} \in \mathbb{R}^7$ and standard deviation $\boldsymbol{\sigma} \in \mathbb{R}^7$ over the dataset, plus physical variances $\boldsymbol{\sigma}_{pos}^2$ and $\boldsymbol{\sigma}_{rot}^2$.

In [ ]:
import numpy as np
import json

# Collect actions across an episode sample (or entire dataset)
sample_indices = min(5000, len(dataset))
print(f"Sampling {sample_indices} action frames to compute empirical normalizer stats...")

actions_list = []
for idx in range(sample_indices):
    actions_list.append(dataset[idx]["action"].numpy())

actions_np = np.stack(actions_list, axis=0)  # [N, 7]

mu_d = np.mean(actions_np, axis=0).tolist()
sigma_d = (np.std(actions_np, axis=0) + 1e-6).tolist()

# Positional variance across 3D translation: (sigma_x^2 + sigma_y^2 + sigma_z^2) / 3
sigma_pos_sq = float(np.mean(np.array(sigma_d[:3]) ** 2))
# Rotational variance across 3D rotation: (sigma_rx^2 + sigma_ry^2 + sigma_rz^2) / 3
sigma_rot_sq = float(np.mean(np.array(sigma_d[3:6]) ** 2))

action_stats = {
    "mu_d": mu_d,
    "sigma_d": sigma_d,
    "sigma_pos_sq": sigma_pos_sq,
    "sigma_rot_sq": sigma_rot_sq
}

print("\n--- Computed Empirical Normalization Statistics ---")
print(json.dumps(action_stats, indent=2))

# Save to JSON for our VGA training pipeline
with open("action_stats.json", "w") as f:
    json.dump(action_stats, f, indent=2)
print("\nSaved action_stats.json successfully!")

### Step 7: Load and Test Existing Baseline (SmolVLA-450M)
Here we load the pre-trained SmolVLA policy from Hugging Face Hub to verify its forward pass and inspect its architecture.

In [ ]:
# 0. Ensure num2words is installed (required by SmolVLM processor)
try:
    import num2words
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "num2words"])
    import importlib
    importlib.invalidate_caches()

# 1. Workaround for Hugging Face @strict decorator incompatibility in Wall-X module
import huggingface_hub.dataclasses
huggingface_hub.dataclasses.strict = lambda cls=None, **kw: (cls if cls is not None else lambda c: c)

# 2. Robust import supporting both new and legacy LeRobot package layouts
try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
except ModuleNotFoundError as e:
    if "lerobot.policies" in str(e):
        from lerobot.common.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    else:
        raise

print("Loading baseline model 'lerobot/smolvla_base'...")
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    baseline_policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
    baseline_policy.to(device)
    baseline_policy.eval()

    # Count total parameters
    total_params = sum(p.numel() for p in baseline_policy.parameters())
    trainable_params = sum(p.numel() for p in baseline_policy.parameters() if p.requires_grad)
    print(f"✅ SmolVLA Base loaded successfully on {device}!")
    print(f"- Total parameters: {total_params:,} (~{total_params / 1e6:.1f}M)")
    print(f"- Trainable parameters: {trainable_params:,}")

    # Test forward pass with sample observation
    with torch.no_grad():
        # Format sample batch
        batch = {k: v.unsqueeze(0).to(device) for k, v in sample.items() if isinstance(v, torch.Tensor)}
        pred_action = baseline_policy.select_action(batch)
        print("\nPredicted Action shape from SmolVLA:", pred_action.shape)
        print("Predicted Action values:", pred_action.cpu().numpy())

except Exception as e:
    print(f"Notice: {e}")

### Step 8: Benchmark Forward Latency of Baseline Policy
We test how fast SmolVLA forward inference is to see if it can run within our 18 ms target.

In [ ]:
import time

if 'baseline_policy' in locals() and torch.cuda.is_available():
    print("Benchmarking SmolVLA inference speed on GPU...")
    # Warmup passes
    for _ in range(20):
        _ = baseline_policy.select_action(batch)

    # Measure 50 iterations with CUDA sync
    torch.cuda.synchronize()
    start_time = time.time()
    for _ in range(50):
        _ = baseline_policy.select_action(batch)
    torch.cuda.synchronize()
    elapsed = (time.time() - start_time) / 50.0 * 1000.0

    print(f"Average SmolVLA Inference Latency: {elapsed:.2f} ms")
    print(f"Control Frequency: {1000.0 / elapsed:.1f} Hz")
else:
    print("Skipping latency test (requires GPU and loaded policy).")